In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

In [2]:
df = pd.read_excel("Superstore Sales Dataset.xlsx", sheet_name="Data")
df.shape

(9800, 19)

In [3]:
print(df.head())

   Row ID        Order ID Order Date  Ship Date Month & Year Order  \
0       1  CA-2017-152156 2017-11-08 2017-11-11      November 2017   
1       2  CA-2017-152156 2017-11-08 2017-11-11      November 2017   
2       3  CA-2017-138688 2017-06-12 2017-06-16          June 2017   
3       4  US-2016-108966 2016-10-11 2016-10-18       October 2016   
4       5  US-2016-108966 2016-10-11 2016-10-18       October 2016   

        Ship Mode Customer ID    Customer Name    Segment        Country  \
0    Second Class    CG-12520      Claire Gute   Consumer  United States   
1    Second Class    CG-12520      Claire Gute   Consumer  United States   
2    Second Class    DV-13045  Darrin Van Huff  Corporate  United States   
3  Standard Class    SO-20335   Sean O'Donnell   Consumer  United States   
4  Standard Class    SO-20335   Sean O'Donnell   Consumer  United States   

              City       State  Postal Code Region       Product ID  \
0        Henderson    Kentucky      42420.0  South 

In [4]:
df.dtypes

Row ID                         int64
Order ID                         str
Order Date            datetime64[us]
Ship Date             datetime64[us]
Month & Year Order               str
Ship Mode                        str
Customer ID                      str
Customer Name                    str
Segment                          str
Country                          str
City                             str
State                            str
Postal Code                  float64
Region                           str
Product ID                       str
Category                         str
Sub-Category                     str
Product Name                     str
Sales                          int64
dtype: object

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Row ID              9800 non-null   int64         
 1   Order ID            9800 non-null   str           
 2   Order Date          9800 non-null   datetime64[us]
 3   Ship Date           9800 non-null   datetime64[us]
 4   Month & Year Order  9800 non-null   str           
 5   Ship Mode           9800 non-null   str           
 6   Customer ID         9800 non-null   str           
 7   Customer Name       9800 non-null   str           
 8   Segment             9800 non-null   str           
 9   Country             9800 non-null   str           
 10  City                9800 non-null   str           
 11  State               9800 non-null   str           
 12  Postal Code         9789 non-null   float64       
 13  Region              9800 non-null   str           
 14  Pro

In [6]:
df.isnull().sum()

Row ID                 0
Order ID               0
Order Date             0
Ship Date              0
Month & Year Order     0
Ship Mode              0
Customer ID            0
Customer Name          0
Segment                0
Country                0
City                   0
State                  0
Postal Code           11
Region                 0
Product ID             0
Category               0
Sub-Category           0
Product Name           0
Sales                  0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.columns = (
    df.columns.str.strip()
    .str.replace(" & ", "_and_", regex=False)
    .str.replace(" ", "_", regex=False)    
    )

df.columns


Index(['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Month_and_Year_Order',
       'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country',
       'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category',
       'Sub-Category', 'Product_Name', 'Sales'],
      dtype='str')

In [9]:
df["Sales"].describe()

count    9.800000e+03
mean     1.137515e+05
std      5.218849e+05
min      3.000000e+00
25%      2.544000e+03
50%      1.110950e+04
75%      5.405700e+04
max      2.396266e+07
Name: Sales, dtype: float64

In [10]:
df["Order_Date"].min(), df["Order_Date"].max()

(Timestamp('2015-01-03 00:00:00'), Timestamp('2018-12-30 00:00:00'))

In [11]:
orders = (
    df.groupby(["Order_ID", "Order_Date", "Customer_ID","Segment",
                "Region", "Ship_Mode"])
    .agg(order_revenue=("Sales", "sum"))
    .reset_index()
)
orders.shape

(4922, 7)

In [12]:
def channel_scorecard(orders, dim):
    g = orders.groupby(dim).agg(
        revenue=("order_revenue", "sum"),
        orders=("Order_ID", "nunique"),
        customers=("Customer_ID", "nunique"),
        avg_order_value=("order_revenue", "mean")
    ).sort_values("revenue", ascending=False)
    g["orders_per_customer"] = (g["orders"] / g["customers"]).round(2)
    g["revenue_share_%"] = (g["revenue"] / g["revenue"].sum() * 100).round(2)
    return g

In [13]:
channel_scorecard(orders, "Segment")

,revenue,orders,customers,avg_order_value,orders_per_customer,revenue_share_%
Segment,,,,,,
Consumer,593177249,2537,409,233810.504139,6.20,53.21
Corporate,329671075,1491,236,221107.360832,6.32,29.57
Home Office,191916475,894,148,214671.672260,6.04,17.22


In [14]:
channel_scorecard(orders, "Region")

,revenue,orders,customers,avg_order_value,orders_per_customer,revenue_share_%
Region,,,,,,
West,360215764,1587,681,226979.057341,2.33,32.31
Central,315500824,1156,626,272924.588235,1.85,28.30
East,287750012,1369,669,210189.928415,2.05,25.81
South,151298199,810,509,186787.900000,1.59,13.57


In [15]:
channel_scorecard(orders, "Ship_Mode")

,revenue,orders,customers,avg_order_value,orders_per_customer,revenue_share_%
Ship_Mode,,,,,,
Standard Class,656654961,2945,774,222972.822071,3.80,58.91
Second Class,225321739,944,545,238688.282839,1.73,20.21
First Class,182269789,772,490,236100.762953,1.58,16.35
Same Day,50518310,261,225,193556.743295,1.16,4.53


In [16]:
orders.shape

(4922, 7)

In [17]:
df.groupby("Category").agg(
    revenue=("Sales", "sum"),
    line_items=("Row_ID", "count"),
    orders=("Order_ID", "nunique"),
    customers=("Customer_ID", "nunique")    
).assign(revenue_share_pct=lambda x: (x["revenue"] / x["revenue"].sum() * 100)
         .round(2)).sort_values("revenue", ascending=False)

,revenue,line_items,orders,customers,revenue_share_pct
Category,,,,,
Furniture,542881104,2078,1727,705,48.70
Technology,321338686,1813,1519,684,28.83
Office Supplies,250545009,5909,3676,787,22.48


In [18]:
subcat = df.groupby(["Category", "Sub-Category"]).agg(
    revenue=("Sales", "sum"),
    line_items=("Row_ID", "count"),
    orders=("Order_ID", "nunique")
    ).reset_index()
subcat["revenue_share_pct"] = (subcat["revenue"] / subcat["revenue"].sum() * 100).round(2)
subcat.sort_values("revenue", ascending=False).head(15)

,Category,Sub-Category,revenue,line_items,orders,revenue_share_pct
0,Furniture,Bookcases,198019806,226,222,17.76
1,Furniture,Chairs,181085736,607,566,16.24
16,Technology,Phones,160430598,876,803,14.39
3,Furniture,Tables,136495172,314,302,12.24
6,Office Supplies,Binders,102579053,1492,1291,9.20
11,Office Supplies,Storage,64148985,832,764,5.75
14,Technology,Copiers,63044102,66,66,5.66
15,Technology,Machines,59424089,115,112,5.33
13,Technology,Accessories,38439897,756,702,3.45
2,Furniture,Furnishings,27280390,931,855,2.45


In [19]:
#seasonality(Monthly and yearly trend of revenue and orders)
monthly = (
    df.assign(Year=df["Order_Date"].dt.year, Month=df["Order_Date"].dt.month)
    .groupby(["Year", "Month"])
    .agg(revenue=("Sales", "sum"), orders=("Order_ID", "nunique"))
    .reset_index()
)
monthly.head(50)

,Year,Month,revenue,orders
0,2015,1,2882798,30
1,2015,2,799427,28
2,2015,3,19553701,69
3,2015,4,8892834,63
4,2015,5,9345661,68
5,2015,6,18025561,64
6,2015,7,16773109,64
7,2015,8,14016061,70
8,2015,9,40800209,129
9,2015,10,14000386,78


In [20]:
df.assign(Year=df["Order_Date"].dt.year) \
.groupby("Year") \
.agg(revenue=("Sales", "sum"), orders=("Order_ID", "nunique"), customers=("Customer_ID", "nunique"))

,revenue,orders,customers
Year,,,
2015,214163636,947,589
2016,242727610,1019,567
2017,305975026,1295,635
2018,351898527,1661,690


In [21]:
#top 10 products by revenue
top_products = (
    df.groupby(["Product_ID", "Product_Name", "Category", "Sub-Category"])
    .agg(revenue=("Sales", "sum"),
         line_items=("Row_ID", "count"),
         orders=("Order_ID", "nunique"))
    .reset_index()
    .sort_values("revenue", ascending=False)
    .head(100)
)
top_products

,Product_ID,Product_Name,Category,Sub-Category,revenue,line_items,orders
49,FUR-BO-10004834,"Riverside Palais Royal Lawyers Bookcase, Royal...",Furniture,Bookcases,30041418,5,5
13,FUR-BO-10001811,"Atlantic Metals Mobile 5-Shelf Bookcases, Cust...",Furniture,Bookcases,23912861,8,8
1639,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,23799932,5,5
5,FUR-BO-10000780,O'Sullivan Plantations 2-Door Library in Landv...,Furniture,Bookcases,23585003,5,5
40,FUR-BO-10004015,"Bush Andora Bookcase, Maple/Graphite Gray Finish",Furniture,Bookcases,18382468,9,9
...,...,...,...,...,...,...,...
28,FUR-BO-10003159,"Sauder Camden County Collection Libraries, Pla...",Furniture,Bookcases,2592799,7,7
89,FUR-CH-10002320,Hon Pagoda Stacking Chairs,Furniture,Chairs,2583889,4,4
1627,TEC-CO-10000971,Hewlett Packard 310 Color Digital Copier,Technology,Copiers,2579914,6,6
360,FUR-TA-10003238,"Chromcraft Bull-Nose Wood 48"" x 96"" Rectangula...",Furniture,Tables,2562057,6,6


In [22]:
#top 10 customers by revenue
cust_lookup = df[["Customer_ID", "Customer_Name"]].drop_duplicates()

top_customers = (
    orders.merge(cust_lookup, on="Customer_ID", how="left")
          .groupby(["Customer_ID", "Customer_Name", "Segment"])
          .agg(total_revenue=("order_revenue", "sum"),
               orders=("Order_ID", "nunique"),
               avg_order_value=("order_revenue", "mean"),
               first_order=("Order_Date", "min"),
               last_order=("Order_Date", "max"))
          .reset_index()
          .sort_values("total_revenue", ascending=False)
          .head(10)
)
top_customers

,Customer_ID,Customer_Name,Segment,total_revenue,orders,avg_order_value,first_order,last_order
24,AG-10675,Anna Gayman,Consumer,24906012,7,3.558002e+06,2016-05-07,2018-09-24
163,CM-12715,Craig Molinari,Corporate,14798847,4,3.699712e+06,2015-03-07,2016-03-01
32,AH-10465,Amy Hunt,Consumer,13672749,5,2.734550e+06,2016-01-03,2018-05-14
602,PO-18850,Patrick O'Brill,Consumer,12682247,11,1.152932e+06,2015-11-02,2018-12-26
275,EP-13915,Emily Phan,Consumer,11809730,17,6.946900e+05,2015-07-06,2018-12-18
730,TA-21385,Tom Ashbrook,Home Office,11388497,4,2.847124e+06,2015-09-12,2018-10-22
328,HM-14860,Harry Marie,Corporate,10803077,10,1.080308e+06,2015-07-21,2018-12-28
705,SO-20335,Sean O'Donnell,Consumer,10279823,6,1.713304e+06,2016-10-11,2018-12-01
136,CC-12670,Craig Carreira,Consumer,10045850,7,1.435121e+06,2015-12-23,2018-12-01
131,CC-12370,Christopher Conant,Consumer,9361086,5,1.872217e+06,2017-05-23,2018-11-17


In [23]:
#how much total revenue comes from the top 10 orders
orders.sort_values("order_revenue", ascending=False).head(10)

,Order_ID,Order_Date,Customer_ID,Segment,Region,Ship_Mode,order_revenue
1799,CA-2017-108987,2017-09-08,AG-10675,Consumer,Central,Second Class,24187328
1316,CA-2016-141243,2016-01-03,AH-10465,Consumer,Central,Second Class,13535016
1218,CA-2016-133585,2016-03-01,CM-12715,Corporate,Central,First Class,12335312
3296,CA-2018-127180,2018-10-22,TA-21385,Home Office,East,First Class,11229902
4512,US-2017-131611,2017-11-05,EP-13915,Consumer,Central,Standard Class,10463124
4295,US-2016-108966,2016-10-11,SO-20335,Consumer,South,Standard Class,9598143
4453,US-2017-110170,2017-09-27,HM-14860,Corporate,Central,Standard Class,9566648
3008,CA-2018-112753,2018-06-18,CC-12670,Consumer,West,Standard Class,9183123
473,CA-2015-139892,2015-09-08,BM-11140,Consumer,Central,Standard Class,8813660
2324,CA-2017-143714,2017-05-23,CC-12370,Consumer,East,Standard Class,8539020


In [24]:
cust = (
    orders.groupby("Customer_ID")
      .agg(
          first_order=("Order_Date", "min"),
          last_order=("Order_Date", "max"),
          total_orders=("Order_ID", "nunique"),
          total_revenue=("order_revenue", "sum"),
          avg_order_value=("order_revenue", "mean")
      )
      .reset_index()
)

cust = cust.merge(
    df[["Customer_ID", "Customer_Name", "Segment", "Region"]].drop_duplicates("Customer_ID"),
    on="Customer_ID", how="left"
)

cust["tenure_days"] = (cust["last_order"] - cust["first_order"]).dt.days
cust["recency_days"] = (df["Order_Date"].max() - cust["last_order"]).dt.days
cust["orders_per_year"] = (cust["total_orders"] / (cust["tenure_days"].replace(0, np.nan) / 365)).round(2)

cust.shape

(793, 12)

In [25]:
cust.head()

,Customer_ID,first_order,last_order,total_orders,total_revenue,avg_order_value,Customer_Name,Segment,Region,tenure_days,recency_days,orders_per_year
0,AA-10315,2015-03-31,2018-06-29,5,5090484,1.018097e+06,Alex Avila,Consumer,Central,1186,184,1.54
1,AA-10375,2015-04-21,2018-12-11,9,165678,1.840867e+04,Allen Armold,Consumer,West,1330,19,2.47
2,AA-10480,2015-05-04,2018-04-15,4,193048,4.826200e+04,Andrew Allen,Consumer,South,1077,259,1.36
3,AA-10645,2015-06-22,2018-11-05,6,1737756,2.896260e+05,Anna Andreadi,Consumer,East,1232,55,1.78
4,AB-10015,2015-02-18,2017-11-10,3,143818,4.793933e+04,Aaron Bergman,Consumer,West,996,415,1.10


In [26]:
cust.describe()

,first_order,last_order,total_orders,total_revenue,avg_order_value,tenure_days,recency_days,orders_per_year
count,793,793,793.000000,7.930000e+02,7.930000e+02,793.000000,793.000000,780.000000
mean,2015-11-14 11:33:40.176544,2018-08-03 17:07:47.591425,6.206810,1.405756e+06,2.256743e+05,993.232030,148.286255,2.972718
min,2015-01-03 00:00:00,2015-10-22 00:00:00,1.000000,1.652000e+03,1.350333e+03,0.000000,0.000000,0.570000
25%,2015-05-16 00:00:00,2018-06-29 00:00:00,4.000000,3.246090e+05,5.886000e+04,820.000000,30.000000,1.680000
50%,2015-09-14 00:00:00,2018-10-16 00:00:00,6.000000,8.095040e+05,1.382971e+05,1088.000000,75.000000,2.225000
75%,2016-01-13 00:00:00,2018-11-30 00:00:00,8.000000,1.713046e+06,2.533732e+05,1218.000000,184.000000,2.912500
max,2018-11-05 00:00:00,2018-12-30 00:00:00,17.000000,2.490601e+07,3.699712e+06,1440.000000,1165.000000,365.000000
std,NaN,NaN,2.525647,1.956416e+06,3.253265e+05,312.210094,187.081466,13.195150


In [27]:
repeat_rate = (cust["total_orders"] > 1).mean() * 100
one_time = (cust["total_orders"] == 1).sum()
repeat = (cust["total_orders"] > 1).sum()

repeat_rate, one_time, repeat

(np.float64(98.36065573770492), np.int64(13), np.int64(780))

In [28]:
cust["total_orders"].value_counts().sort_index()

total_orders
1      13
2      34
3      55
4     106
5     136
6     112
7     111
8      71
9      70
10     42
11     20
12     18
13      4
17      1
Name: count, dtype: int64

In [29]:
cust.assign(buyer_type=np.where(cust["total_orders"] > 1, "Repeat", "One-time")) \
    .groupby("buyer_type").agg(
        customers=("Customer_ID", "count"),
        revenue=("total_revenue", "sum"),
        median_revenue=("total_revenue", "median"),
        median_aov=("avg_order_value", "median")
    )

,customers,revenue,median_revenue,median_aov
buyer_type,,,,
One-time,13,1842068,14112.0,14112.000000
Repeat,780,1112922731,832387.5,139570.471429


In [30]:
#RFM segmentation
rfm = cust[["Customer_ID", "Customer_Name", "Segment", "Region",
            "recency_days", "total_orders", "total_revenue"]].copy()

rfm["R_score"] = pd.qcut(rfm["recency_days"], 5,
                         labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["total_orders"].rank(method="first"), 5,
                         labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["total_revenue"].rank(method="first"), 5,
                         labels=[1, 2, 3, 4, 5]).astype(int)

rfm["RFM_score"] = rfm["R_score"].astype(str) + \
                   rfm["F_score"].astype(str) + \
                   rfm["M_score"].astype(str)

rfm.head()


,Customer_ID,Customer_Name,Segment,Region,recency_days,total_orders,total_revenue,R_score,F_score,M_score,RFM_score
0,AA-10315,Alex Avila,Consumer,Central,184,5,5090484,2,2,5,225
1,AA-10375,Allen Armold,Consumer,West,19,9,165678,5,5,1,551
2,AA-10480,Andrew Allen,Consumer,South,259,4,193048,1,1,1,111
3,AA-10645,Anna Andreadi,Consumer,East,55,6,1737756,3,3,4,334
4,AB-10015,Aaron Bergman,Consumer,West,415,3,143818,1,1,1,111


In [31]:
def rfm_segment(row):
    r, f, m = row["R_score"], row["F_score"], row["M_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    if r >= 3 and f >= 3 and m >= 3:
        return "Loyal"
    if r >= 4 and f <= 2:
        return "New / Promising"
    if r <= 2 and f >= 4 and m >= 4:
        return "At-Risk High Value"
    if r <= 2 and f >= 3:
        return "At-Risk"
    if r <= 2 and f <= 2:
        return "Lost / Hibernating"
    return "Needs Attention"

rfm["RFM_Segment"] = rfm.apply(rfm_segment, axis=1)
rfm["RFM_Segment"].value_counts()

RFM_Segment
Lost / Hibernating    175
Loyal                 146
Needs Attention       140
Champions             102
At-Risk                95
New / Promising        89
At-Risk High Value     46
Name: count, dtype: int64

In [32]:
rfm.groupby("RFM_Segment").agg(
    customers=("Customer_ID", "count"),
    revenue=("total_revenue", "sum"),
    median_recency=("recency_days", "median"),
    median_orders=("total_orders", "median"),
    median_revenue=("total_revenue", "median")
).assign(
    pct_customers=lambda x: (x["customers"] / x["customers"].sum() * 100).round(1),
    pct_revenue=lambda x: (x["revenue"] / x["revenue"].sum() * 100).round(1)
).sort_values("revenue", ascending=False)

,customers,revenue,median_recency,median_orders,median_revenue,pct_customers,pct_revenue
RFM_Segment,,,,,,,
Loyal,146,299702464,52.5,7.0,1255399.0,18.4,26.9
Champions,102,298780790,28.0,9.0,2199527.5,12.9,26.8
Lost / Hibernating,175,157658474,307.0,4.0,401021.0,22.1,14.1
At-Risk High Value,46,119799976,159.5,9.0,1905825.5,5.8,10.7
At-Risk,95,85502289,183.0,6.0,697874.0,12.0,7.7
Needs Attention,140,80752632,57.0,6.0,415429.5,17.7,7.2
New / Promising,89,72568174,26.0,4.0,420692.0,11.2,6.5


In [33]:
#save the outputs to CSV files
import os
os.makedirs("outputs", exist_ok=True)

orders.to_csv("outputs/orders.csv", index=False)
cust.to_csv("outputs/customers.csv", index=False)
rfm.to_csv("outputs/rfm.csv", index=False)
subcat.to_csv("outputs/subcategory.csv", index=False)
monthly.to_csv("outputs/monthly.csv", index=False)

print("Saved.")

Saved.
